# Alpine Crest Private-Banking Transaction Fraud Baseline

This notebook introduces the first **Alpine Crest Private Bank** private-banking scenario. The task is to score transaction alerts using **Banking relationship**, account, relationship-manager, and role context, then read the result with alert-aware metrics.

In [ ]:
import pandas as pd

from banking_fraud_lab import (
    build_learner_facing_views,
    evaluate_alert_scores,
    generate_private_banking_transaction_fraud_world,
    load_tables_to_sqlite,
)
from banking_fraud_lab.generators.private_banking import PRIVATE_BANKING_ACTIVITY_TYPE
from banking_fraud_lab.schema import PROTECTED_SCENARIO_ANSWER_KEYS

## Generate The Scenario

The generator injects a configurable transaction-fraud pattern into Alpine Crest records. Learner-facing views include cases and outcomes for model evaluation, but do not expose protected answer keys.

In [ ]:
tables = generate_private_banking_transaction_fraud_world(
    seed=42,
    scenario_prevalence=0.2,
)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables

pd.DataFrame(
    [
        {"table_name": table_name, "row_count": len(frame)}
        for table_name, frame in learner_tables.items()
    ]
)

## Build Private-Banking Features

The feature query keeps the case outcome separate from the fields used by the scoring rule. It joins alerts to transactions, accounts, relationship-manager assignment, and role counts under the **Banking relationship**.

In [ ]:
connection = load_tables_to_sqlite(learner_tables, ":memory:")
private_cases = pd.read_sql_query(
    """
    WITH role_counts AS (
        SELECT
            banking_relationship_id,
            COUNT(*) AS relationship_role_count
        FROM partner_roles
        GROUP BY banking_relationship_id
    )
    SELECT
        c.case_id,
        c.alert_id,
        al.alert_type,
        al.severity,
        t.transaction_id,
        CAST(t.amount_chf AS REAL) AS amount_chf,
        CAST(a.balance_chf AS REAL) AS balance_chf,
        br.banking_relationship_id,
        br.relationship_manager_code,
        rc.relationship_role_count,
        CAST(co.confirmed_fraud AS INTEGER) AS confirmed_fraud,
        CAST(co.loss_amount_chf AS REAL) AS loss_amount_chf
    FROM cases AS c
    JOIN alerts AS al
        ON c.alert_id = al.alert_id
    JOIN transactions AS t
        ON c.transaction_id = t.transaction_id
    JOIN accounts AS a
        ON c.account_id = a.account_id
    JOIN banking_relationships AS br
        ON c.banking_relationship_id = br.banking_relationship_id
    JOIN role_counts AS rc
        ON br.banking_relationship_id = rc.banking_relationship_id
    JOIN case_outcomes AS co
        ON c.case_id = co.case_id
    WHERE al.institution_name = 'Alpine Crest Private Bank'
    ORDER BY c.opened_at
    """,
    connection,
)

assert not private_cases.empty
assert private_cases["confirmed_fraud"].nunique() == 2
private_cases

## Score Alerts

This baseline uses a transparent scoring rule, not a trained production model. It raises scores for the injected private-banking fraud alert type, larger amount-to-balance ratios, and richer Banking relationship role context.

In [ ]:
features = private_cases.copy()
features["amount_to_balance"] = (
    features["amount_chf"] / features["balance_chf"].clip(lower=1.0)
)
features["score"] = (
    0.50 * (features["amount_to_balance"] * 3).clip(upper=1.0)
    + 0.35 * features["alert_type"].eq(PRIVATE_BANKING_ACTIVITY_TYPE).astype(float)
    + 0.15 * (features["relationship_role_count"] >= 2).astype(float)
).clip(upper=1.0)

features[
    [
        "case_id",
        "alert_type",
        "amount_chf",
        "balance_chf",
        "amount_to_balance",
        "relationship_role_count",
        "score",
        "confirmed_fraud",
    ]
]

## Evaluate Threshold Tradeoffs

Alert-aware metrics show the review workload and fraud-detection tradeoff at each threshold. The report deliberately avoids headline accuracy because case outcomes are sparse and operationally mediated.

In [ ]:
report = evaluate_alert_scores(
    features[["case_id", "alert_id"]],
    features[["case_id", "confirmed_fraud", "loss_amount_chf"]],
    features[["alert_id", "score"]],
    thresholds=(0.35, 0.55, 0.75),
    alert_capacity=2,
    investigation_cost_chf=100.0,
    false_positive_cost_chf=25.0,
)

threshold_summary = pd.DataFrame(report["threshold_summaries"])
threshold_summary.insert(0, "pr_auc", report["pr_auc"])
assert "accuracy is out of scope" in report["limitation_summary"]
threshold_summary[
    [
        "pr_auc",
        "threshold",
        "precision",
        "recall",
        "alert_volume",
        "alert_capacity",
        "over_capacity_alerts",
        "total_cost_chf",
    ]
]

In [ ]:
connection.close()